# sketch2stl · ML1 — Stroke recogniser (trained from scratch)

**Question it answers:** *what kind of stroke did the person just draw?* → `line | arc | circle | rect | polyline` (+ confidence)

**Model category:** trained from scratch — a `HistGradientBoostingClassifier` over 12 geometric features of the stroke.

| | |
|---|---|
| Training data | 14,608 synthetic hand-drawn strokes: Fusion 360 Gallery CAD curves re-drawn by the repo's calibrated *mouse* hand model (`synthetic_mouse` in the dataset) |
| Held-out real test | 325 strokes hand-drawn by PK (mouse, 2 sessions) — **never trained on** |
| Split | by CAD model, official `train_test.json` (3,000 held-out test strokes) |
| Baseline | geometric rules (least-squares circle / line / rectangle fits) on the same strokes |

Dataset: [pakiino/sketch2stl-datasets](https://huggingface.co/datasets/pakiino/sketch2stl-datasets) · Code: [SunKaiwen1023/sketch2stl](https://github.com/SunKaiwen1023/sketch2stl)

**Run all.** No GPU needed (about 3–5 minutes on the free Colab CPU). The last section pushes the trained model to the Hugging Face Hub — it only runs when you set `PUSH_TO_HUB = True`.


## 1 · Setup

In [ ]:
import os, sys, subprocess
REPO_URL  = "https://github.com/SunKaiwen1023/sketch2stl"
REPO_DIR  = "sketch2stl"
DATASET   = "pakiino/sketch2stl-datasets"
MODEL_REPO = "pakiino/sketch2stl-recognizer"     # where the trained model is pushed
PUSH_TO_HUB = False                              # set True when you are happy with the numbers

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "shapely>=2.0", "scikit-image", "joblib", "huggingface_hub"], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, os.getcwd())
print("working in", os.getcwd())

## 2 · Download the dataset from Hugging Face

In [ ]:
from huggingface_hub import snapshot_download
LOCAL = os.environ.get("SKETCH2STL_DATA")          # only for offline testing
root = LOCAL or snapshot_download(DATASET, repo_type="dataset",
                                  allow_patterns=["sketch2stl_datasets/ML1_stroke_recognizer/synthetic_mouse/*", "sketch2stl_datasets/ML1_stroke_recognizer/manual/*"])
D = os.path.join(root, "sketch2stl_datasets", "ML1_stroke_recognizer") if not LOCAL else LOCAL
for dp, _, fs in os.walk(D):
    for f in fs: print(os.path.relpath(os.path.join(dp, f), D))

## 3 · Synthetic strokes (train / test)

Made by the repo's `scripts/build_dataset.py --style mouse` from the Fusion 360 Gallery: every clean CAD curve is
re-drawn by `sketch2stl.data.synth.handdraw` with per-class mouse tremor. Already in the trainer's format
(64 points per stroke, mm), split by CAD model.

In [ ]:
import numpy as np, shutil
from collections import Counter
os.makedirs("data/strokes", exist_ok=True)
for name in ("train", "test"):
    shutil.copy(os.path.join(D, "synthetic_mouse", f"{name}.npz"), f"data/strokes/{name}.npz")
    z = np.load(f"data/strokes/{name}.npz", allow_pickle=True)
    print(f"{name:5s} {len(z['labels']):6d} strokes  {len(set(z['model_ids'])):5d} CAD models  {dict(Counter(z['labels'].astype(str)))}")

## 4 · Real hand-drawn strokes → held-out test

Imported with the repo's own `scripts/import_strokes.py` (same `prepare_points`). These are the **manual dataset** and are used **only** to test — they answer *does training on synthetic strokes transfer to a real hand?*

In [ ]:
subprocess.run([sys.executable, "scripts/import_strokes.py",
                "--data", os.path.join(D, "manual", "hand_drawn"), "--out", "data/real"], check=True)
r = np.load("data/real/real_all.npz", allow_pickle=True)
print(len(r["labels"]), "real strokes", dict(Counter(r["labels"].astype(str))))

## 5 · Train + the three-arm comparison

One command, the repo's `scripts/train_recognizer.py`:

* **manual** — the user picks the tool: perfect by definition, the ceiling
* **rules** — least-squares fits, no training: the baseline to beat
* **ml** — the HistGradientBoosting model

Scored on the same held-out synthetic strokes, then on the 325 real strokes.

In [ ]:
res = subprocess.run([sys.executable, "scripts/train_recognizer.py", "--data", "data/strokes",
                      "--out", "models/recognizer", "--real-test", "data/real/real_all.npz"],
                     capture_output=True, text=True)
print(res.stdout[-6000:]); print(res.stderr[-3000:])

## 6 · Results

In [ ]:
import json, pandas as pd
R = json.load(open("models/recognizer/results.json"))
table = pd.DataFrame({
    "synthetic test · rules": R["rules"], "synthetic test · ml": R["ml"],
    "REAL strokes · rules": R.get("rules_on_real", {}), "REAL strokes · ml": R.get("ml_on_real", {}),
}).loc[["n", "accuracy", "macro_f1"]].T
per = pd.DataFrame({k: v["per_class_f1"] for k, v in {
    "synthetic · rules": R["rules"], "synthetic · ml": R["ml"],
    "REAL · rules": R["rules_on_real"], "REAL · ml": R["ml_on_real"]}.items()}).T
display(table); print("per-class F1"); display(per.round(3))
print(f"ml - rules (synthetic) macro-F1: {R['gap_macro_f1']:+.3f}")
print(f"synthetic -> real drop for ml:   {R.get('domain_gap_macro_f1', float('nan')):+.3f}")

In [ ]:
from IPython.display import Image as IPImage
IPImage("models/recognizer/confusion.png")

### Real-stroke confusion

Where the model goes wrong on a real hand (rows = true, columns = predicted).

In [ ]:
import matplotlib.pyplot as plt
LABELS = ["line", "arc", "circle", "rect", "polyline"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, key in zip(axes, ["rules_on_real", "ml_on_real"]):
    cm = np.array(R[key]["confusion"]); ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(5)); ax.set_xticklabels(LABELS, rotation=40); ax.set_yticks(range(5)); ax.set_yticklabels(LABELS)
    for i in range(5):
        for j in range(5): ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
    ax.set_title(f"{key.replace('_', ' ')} · macro-F1 {R[key]['macro_f1']:.3f}")
plt.tight_layout(); plt.savefig("models/recognizer/confusion_real.png", dpi=140); plt.show()

## 7 · Sanity check: the app's loader can use it

In [ ]:
from sketch2stl.recognizer.ml import MLRecognizer
from sketch2stl.types import Stroke
from sketch2stl.strokes import mm_to_px
rec = MLRecognizer("models/recognizer")
assert rec.model is not None, "model did not load"
for k in range(5):
    pts, lab = r["strokes"][k * 60], str(r["labels"][k * 60])
    p = rec.recognize(Stroke(points=mm_to_px(pts)))
    print(f"true {lab:8s} -> {p.kind.value:8s} conf {p.confidence:.2f}")

## 8 · Push the model to the Hugging Face Hub

Writes a model card with the numbers above and uploads `model.joblib`, `results.json` and the figures to `MODEL_REPO`. The app downloads it from there, so everyone (Colab, laptops, the Space) runs the same model.

Set `PUSH_TO_HUB = True` in section 1 (or here), then run. You will be asked for a Hugging Face **write** token.

In [ ]:
PUSH_TO_HUB = PUSH_TO_HUB  # flip to True here if you prefer
card = f"""---
license: other
license_name: fusion-360-gallery-dataset-license
license_link: https://github.com/AutodeskAILab/Fusion360GalleryDataset
library_name: sklearn
tags: [sketch, cad, stroke-recognition, tabular-classification]
datasets: [{DATASET}]
---
# sketch2stl · stroke recogniser (ML1)

Classifies one hand-drawn stroke as **line, arc, circle, rect or polyline** for the sketch2stl app
(draw a 2-D shape → printable STL). Trained **from scratch**: `HistGradientBoostingClassifier` over 12
geometric features (`sketch2stl.recognizer.features`).

## Training data
14,608 synthetic hand-drawn strokes (`ML1_stroke_recognizer/synthetic_mouse`): Fusion 360 Gallery CAD curves
re-drawn by the repo's mouse hand model, split by CAD model.

## Evaluation
| test set | n | rules macro-F1 | **ml macro-F1** |
|---|---|---|---|
| synthetic, held-out projects | {R['ml']['n']} | {R['rules']['macro_f1']:.3f} | **{R['ml']['macro_f1']:.3f}** |
| **real hand-drawn (PK, mouse)** | {R['ml_on_real']['n']} | {R['rules_on_real']['macro_f1']:.3f} | **{R['ml_on_real']['macro_f1']:.3f}** |

Per-class F1 on real strokes: {R['ml_on_real']['per_class_f1']}

## Intended use
Inside the sketch2stl app, loaded with `MLRecognizer("model.joblib")`. The user always sees what was
recognised and can override it. Not for other drawing styles without re-checking: all real test strokes
come from one person on a mouse.

## Licence
Derived from the Fusion 360 Gallery Dataset (Autodesk) — non-commercial research use only.
"""
open("models/recognizer/README.md", "w").write(card)
print(card[:1500])

if PUSH_TO_HUB:
    from huggingface_hub import notebook_login, create_repo, upload_folder
    notebook_login()
    create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    upload_folder(repo_id=MODEL_REPO, folder_path="models/recognizer", repo_type="model",
                  commit_message="ML1 stroke recogniser + model card")
    print("pushed -> https://huggingface.co/" + MODEL_REPO)
else:
    print("PUSH_TO_HUB is False - nothing uploaded.")